In [1]:
!pip install -q "transformers>=4.51.0" datasets torch accelerate peft sentencepiece protobuf


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import json
import re
import os
import random
import time
import gc
from datetime import datetime, timedelta
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType

# ============================================================
# CONFIGURATION
# ============================================================
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_NAME   = "Qwen/Qwen3-1.7B"
MAX_LENGTH   = 1024
MAX_NEW_TOKENS = 2048

# Training hyperparameters
LEARNING_RATE    = 2e-5
WEIGHT_DECAY     = 0.01
LORA_RANK        = 16
LORA_ALPHA       = 32
LORA_DROPOUT     = 0.1
NUM_TRAIN        = 1000
NUM_TEST         = 100
NUM_EPOCHS       = 3
GRAD_ACCUM_STEPS = 8
MAX_GRAD_NORM    = 1.0
WARMUP_RATIO     = 0.1

# Composite loss weights (following LLM-JEPA defaults)
GAMMA   = 1.0   # CE loss weight
LAMBDA_ = 0.1   # Cosine similarity (JEPA) loss weight
LAST_TOKEN = -3   # Qwen offset to skip <|im_end|> and trailing tokens


# JEPA predictor tokens (following LLM-JEPA architecture)
NUM_PREDICTORS = 1   

# Language
LANG_CONFIG = "pa"   # Punjabi native script
LANG_NAME   = "Punjabi"

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

PROMPT_TEMPLATE = (
    "Solve this math problem. Show your work step by step "
    "and give the final numeric answer.\n\nQuestion: {question}"
)

# Derived
steps_per_epoch = NUM_TRAIN // GRAD_ACCUM_STEPS
total_steps     = steps_per_epoch * NUM_EPOCHS
warmup_steps    = int(total_steps * WARMUP_RATIO)

print("=" * 70)
print("  INDIC-JEPA: Composite Loss SFT on Qwen3-1.7B")
print("=" * 70)
print(f"  Model:                {MODEL_NAME}")
print(f"  Language:             {LANG_NAME} ({LANG_CONFIG})")
print(f"  Train / Test:         {NUM_TRAIN} / {NUM_TEST}")
print(f"  Epochs:               {NUM_EPOCHS}")
print(f"  Grad accumulation:    {GRAD_ACCUM_STEPS}")
print(f"  Effective batch size: {GRAD_ACCUM_STEPS}")
print(f"  Steps per epoch:      {steps_per_epoch}")
print(f"  Total optim. steps:   {total_steps}")
print(f"  Warmup steps:         {warmup_steps}")
print(f"  LR:                   {LEARNING_RATE}")
print(f"  Loss:                 {GAMMA}·CE + {LAMBDA_}·(1 − CosSim)")
print(f"  Predictor tokens:     {NUM_PREDICTORS} (after PA answer)")
print(f"  LoRA:                 rank={LORA_RANK} α={LORA_ALPHA} drop={LORA_DROPOUT}")
print(f"  Forward passes/sample:3 (PA full CE + PA Q+pred + EN answer)")
print("=" * 70)



  INDIC-JEPA: Composite Loss SFT on Qwen3-1.7B
  Model:                Qwen/Qwen3-1.7B
  Language:             Punjabi (pa)
  Train / Test:         500 / 100
  Epochs:               3
  Grad accumulation:    8
  Effective batch size: 8
  Steps per epoch:      62
  Total optim. steps:   186
  Warmup steps:         18
  LR:                   2e-05
  Loss:                 1.0·CE + 0.1·(1 − CosSim)
  Predictor tokens:     1 (appended to Punjabi prompt)
  Last token offset:    -3 (Qwen convention)
  LoRA:                 rank=16 α=32 drop=0.1
  Forward passes/sample:2 (Punjabi w/grad + English w/grad)


In [ ]:
# ============================================================
# LOAD MODEL + LoRA + PREDICTOR TOKENS
# ============================================================
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Add predictor special tokens (following LLM-JEPA)
pred_tokens = [f"<|predictor_{i}|>" for i in range(1, NUM_PREDICTORS + 1)]
new_tokens = [t for t in pred_tokens if t not in tokenizer.get_vocab()]
if new_tokens:
    tokenizer.add_special_tokens({"additional_special_tokens": new_tokens})
    print(f"Added {len(new_tokens)} predictor tokens: {new_tokens}")

PRED_TOKEN_IDS = [tokenizer.convert_tokens_to_ids(t) for t in pred_tokens]
print(f"Predictor token IDs: {PRED_TOKEN_IDS}")

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    use_cache=False,
)

# Resize embeddings for new predictor tokens
if new_tokens:
    model.resize_token_embeddings(len(tokenizer))
    print(f"Resized embeddings → {len(tokenizer)} tokens")

print("Applying LoRA adapters...")
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)
model = get_peft_model(model, lora_config)
model.enable_input_require_grads()
model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)
model.print_trainable_parameters()

device = next(model.parameters()).device
print(f"Device: {device}")
print(f"Dtype:  {next(model.parameters()).dtype}")


Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Resized embeddings → 151670 tokens
Applying LoRA adapters...
trainable params: 17,432,576 || all params: 1,737,462,784 || trainable%: 1.0033
Device: cuda:0
Dtype:  torch.bfloat16


In [4]:
# ============================================================
# LOAD PAIRED DATASET (sarvamai/gsm8k-indic, Punjabi config)
# Each row contains BOTH the Punjabi question AND the original
# English question, so we get perfectly aligned pairs for free.
# ============================================================
print(f"Loading sarvamai/gsm8k-indic  config={LANG_CONFIG}...")
indic_ds = load_dataset("sarvamai/gsm8k-indic", LANG_CONFIG, split="test")
print(f"Total samples: {len(indic_ds)}")
print(f"Fields: {indic_ds.column_names}")

# Shuffle deterministically and split
all_indices = list(range(len(indic_ds)))
random.Random(SEED).shuffle(all_indices)

train_indices = all_indices[:NUM_TRAIN]
test_indices  = all_indices[NUM_TRAIN : NUM_TRAIN + NUM_TEST]
assert len(set(train_indices) & set(test_indices)) == 0, "Leak!"

train_samples = []
for idx in train_indices:
    r = indic_ds[idx]
    train_samples.append({
        "pa_question": r["question"],
        "pa_answer":   r["answer"],
        "en_question":  r["original_question"],
        "en_answer":    r["original_answer"],
    })

test_samples = []
for idx in test_indices:
    r = indic_ds[idx]
    test_samples.append({
        "pa_question": r["question"],
        "pa_answer":   r["answer"],
        "en_question":  r["original_question"],
        "en_answer":    r["original_answer"],
    })

print(f"Train: {len(train_samples)} paired (PA↔EN) samples")
print(f"Test:  {len(test_samples)} samples (unseen)")

# --- Verify pairing ---
print("\n--- Pairing verification (first 3) ---")
for i in range(3):
    s = train_samples[i]
    en_num = re.search(r'####\s*(-?[\d,]+\.?\d*)', s["en_answer"])
    pa_num = re.search(r'####\s*(-?[\d,]+\.?\d*)', s["pa_answer"])
    en_n = en_num.group(1) if en_num else "?"
    pa_n = pa_num.group(1) if pa_num else "?"
    print(f"[{i}] EN: {s['en_question'][:60]}…  →  {en_n}")
    print(f"    PA: {s['pa_question'][:60]}…  →  {pa_n}")
    print(f"    Match: {en_n == pa_n}")


Loading sarvamai/gsm8k-indic  config=pa...


README.md: 0.00B [00:00, ?B/s]

pa/test-00000-of-00001.parquet:   0%|          | 0.00/711k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Total samples: 1319
Fields: ['original_question', 'answer', 'question', 'original_answer']
Train: 500 paired (PA↔EN) samples
Test:  100 samples (unseen)

--- Pairing verification (first 3) ---
[0] EN: Jared is trying to increase his typing speed. He starts with…  →  ?
    PA: ਜੇਰਡ ਆਪਣੀ ਟਾਈਪਿੰਗ ਸਪੀਡ ਵਧਾਉਣ ਦੀ ਕੋਸ਼ਿਸ਼ ਕਰ ਰਿਹਾ ਹੈ। ਉਹ 47 ਸ਼…  →  52
    Match: False
[1] EN: Jordan has 2 children who wear diapers.  Each child requires…  →  ?
    PA: ਜੌਰਡਨ ਦੇ 2 ਬੱਚੇ ਹਨ ਜੋ ਡਾਇਪਰ ਪਹਿਨਦੇ ਹਨ। ਹਰ ਬੱਚੇ ਨੂੰ ਪ੍ਰਤੀ ਦਿਨ…  →  5
    Match: False
[2] EN: A wooden bridge can carry no more than 5000 pounds. A delive…  →  ?
    PA: ਇੱਕ ਲੱਕੜੀ ਦਾ ਪੁਲ 5000 ਪੌਂਡ ਤੋਂ ਵੱਧ ਭਾਰ ਨਹੀਂ ਚੁੱਕ ਸਕਦਾ। ਇੱਕ ਡ…  →  83
    Match: False


In [ ]:
# ============================================================
# HELPERS
# ============================================================

def extract_gold_answer(answer_str):
    """Extract numeric answer from GSM8K '#### N' format."""
    m = re.search(r'####\s*(-?[\d,]+\.?\d*)', answer_str)
    return m.group(1).replace(",", "").strip() if m else None

def extract_model_answer(text):
    """Extract last numeric value from model output."""
    boxed = re.findall(r'\\boxed\{([^}]+)\}', text)
    if boxed:
        n = re.sub(r'[^\d.\-]', '', boxed[-1])
        if n: return n
    for pat in [
        r'(?:answer|result|total)\s*(?:is|=|:)\s*(-?[\d,]+\.?\d*)',
        r'####\s*(-?[\d,]+\.?\d*)',
        r'=\s*(-?[\d,]+\.?\d*)\s*$',
    ]:
        ms = re.findall(pat, text, re.I | re.M)
        if ms: return ms[-1].replace(",", "").strip()
    nums = re.findall(r'-?[\d,]+\.?\d*', text)
    return nums[-1].replace(",", "").strip() if nums else None

def compare_answers(pred, gold):
    if pred is None or gold is None: return False
    try:    return abs(float(pred) - float(gold)) < 1e-3
    except: return pred.strip() == gold.strip()

def find_pred_token_pos(input_ids):
    """Find position of the LAST predictor token in the sequence."""
    pred_id = PRED_TOKEN_IDS[-1]
    positions = (input_ids[0] == pred_id).nonzero(as_tuple=True)[0]
    assert len(positions) > 0, f"Predictor token {pred_id} not found!"
    return positions[-1].item()

def find_en_hidden_pos(input_ids):
    """Find the hidden state extraction position for EN answer.
    Uses LAST_TOKEN offset from unpadded length, matching LLM-JEPA's
    _last_token_index for Qwen (offset=-3)."""
    seq_len = input_ids.shape[1]  # no padding in our setup
    return seq_len + LAST_TOKEN


def build_prompts(sample):
    """Return 4 prompts for one training sample.

    pa_full_text    : PA Q + PA answer       → CE loss (forward pass 1)
    pa_q_pred_text  : PA Q + [PRED] tokens   → h_pa    (forward pass 2)
    en_answer_text  : EN answer standalone    → h_en    (forward pass 3)
    pa_q_only_text  : PA Q only              → for label masking length
    """
    pa_q = PROMPT_TEMPLATE.format(question=sample["pa_question"])
    

    # 1) PA full: question + answer (for CE loss)
    pa_full_text = tokenizer.apply_chat_template(
        [{"role": "user",      "content": pa_q},
         {"role": "assistant", "content": sample["pa_answer"]}],
        tokenize=False, add_generation_prompt=False, enable_thinking=False,
    )

    # 2) PA question + predictor tokens (source for JEPA)
    #    Pred tokens appended to the question content, matching LLM-JEPA
    pa_q_with_pred = pa_q
    for k in range(NUM_PREDICTORS, 0, -1):
        pa_q_with_pred += f"<|predictor_{k}|>"
    pa_q_pred_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": pa_q_with_pred}],
        tokenize=False, add_generation_prompt=False, enable_thinking=False,
    )

    # 3) EN answer standalone (target for JEPA)
    #    Encoded as an assistant message, matching LLM-JEPA's
    #    get_assistant_messages → apply_chat_template
    en_answer_text = tokenizer.apply_chat_template(
        [{"role": "assistant", "content": sample["en_answer"]}],
        tokenize=False, add_generation_prompt=False, enable_thinking=False,
    )

    # 4) PA question only (for label mask length)
    pa_q_only_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": pa_q}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )

    return pa_full_text, pa_q_pred_text, en_answer_text, pa_q_only_text

# ---- Verification ----
s0 = train_samples[0]
_pa_full, _pa_qp, _en_ans, _pa_q = build_prompts(s0)

_pa_full_ids = tokenizer.encode(_pa_full)
_pa_qp_ids   = tokenizer.encode(_pa_qp)
_en_ans_ids  = tokenizer.encode(_en_ans)
_pa_q_ids    = tokenizer.encode(_pa_q)

# Verify question prefix for label masking
assert _pa_full_ids[:len(_pa_q_ids)] == _pa_q_ids, "PA question prefix mismatch!"

# Verify pred token is in the question+pred sequence
_pa_qp_enc = tokenizer(_pa_qp, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
pred_pos = find_pred_token_pos(_pa_qp_enc["input_ids"])

# Verify EN answer encoding
_en_ans_enc = tokenizer(_en_ans, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)


# Verify EN answer extraction position (using LAST_TOKEN offset)
h_en_idx = find_en_hidden_pos(_en_ans_enc["input_ids"])

print("✓ Prompt structure verified.")
print(f"  PA Q-only tokens       : {len(_pa_q_ids)}")
print(f"  PA full (Q+A) tokens   : {len(_pa_full_ids)}")
print(f"  PA Q+pred tokens       : {len(_pa_qp_ids)}  (pred at position {pred_pos})")
print(f"  EN answer-only tokens  : {len(_en_ans_ids)}  (h_en at position {h_en_idx}, offset={LAST_TOKEN})")
print(f"  Answer tokens (PA)     : {len(_pa_full_ids) - len(_pa_q_ids)}")
print(f"\n  Token at pred pos ({pred_pos}): '{tokenizer.decode(_pa_qp_enc['input_ids'][0, pred_pos])}'")
print(f"  Token at h_en pos ({h_en_idx}): '{tokenizer.decode(_en_ans_enc['input_ids'][0, h_en_idx])}'")
print(f"\n  PA Q+pred decoded (last 30 chars): ...{_pa_qp[-30:]}")
print(f"  EN answer decoded (first 80 chars): {_en_ans[:80]}...")


✓ Prompt structure verified.
  PA question tokens     : 470
  PA full tokens         : 589
  PA+predictor tokens    : 464  (+ 1 pred tokens)
  EN question tokens     : 94
  Answer tokens          : 119

  Predictor token IDs: [151669]
  Last 5 tokens of PA+pred: [122, 30, 151669, 151645, 198]
  Decoded: �?<|predictor_1|><|im_end|>


  h_pa extraction position : 461 (of 464 tokens)
  h_en extraction position : 91 (of 94 tokens)
  Token at h_pa pos: <|predictor_1|>
  Token at h_en pos: ?


In [ ]:
# ============================================================
# BASELINE: cosine similarity BEFORE training
# Source: PA question + [PRED]  |  Target: EN answer
# ============================================================
NUM_BASELINE = 50
print(f"Computing pre-training cosine similarities ({NUM_BASELINE} samples)...")
print(f"  Source: PA question + [PRED]  →  Target: EN answer (standalone)")
model.eval()
baseline_cos_sims = []

with torch.no_grad():
    for i in range(NUM_BASELINE):
        s = train_samples[i]
        _, pa_qp, en_ans, _ = build_prompts(s)

        pa_enc = tokenizer(pa_qp,  return_tensors="pt",
                           truncation=True, max_length=MAX_LENGTH).to(device)
        en_enc = tokenizer(en_ans, return_tensors="pt",
                           truncation=True, max_length=MAX_LENGTH).to(device)

        out_pa = model(**pa_enc, output_hidden_states=True)
        out_en = model(**en_enc, output_hidden_states=True)

        h_pa = out_pa.hidden_states[-1][0, find_pred_token_pos(pa_enc["input_ids"]), :]
        h_en = out_en.hidden_states[-1][0, find_en_hidden_pos(en_enc["input_ids"]), :]
        cs = F.cosine_similarity(
            h_pa.unsqueeze(0).float(),
            h_en.unsqueeze(0).float(), dim=-1
        ).item()
        baseline_cos_sims.append(cs)

baseline_mean = sum(baseline_cos_sims) / len(baseline_cos_sims)
baseline_std  = (sum((x - baseline_mean)**2 for x in baseline_cos_sims)
                 / len(baseline_cos_sims)) ** 0.5
print(f"\nPre-training PA_Q+[PRED] ↔ EN_Answer cosine similarity:")
print(f"  Mean: {baseline_mean:.4f}  (std {baseline_std:.4f})")
print(f"  Min:  {min(baseline_cos_sims):.4f}")
print(f"  Max:  {max(baseline_cos_sims):.4f}")


Computing pre-training cosine similarities (50 samples)...

Pre-training PA↔EN cosine similarity:
  Mean: 0.1213  (std 0.0406)
  Min:  0.0480
  Max:  0.2794


In [ ]:
# ============================================================
# TRAINING LOOP — Faithful LLM-JEPA architecture
#   Pass 1: PA Q + PA answer     → CE loss on answer tokens
#   Pass 2: PA Q + [PRED]        → h_pa at pred token
#           (pred only sees question — must PREDICT answer repr)
#   Pass 3: EN answer standalone  → h_en at last token
#   All 3 with gradient (unfrozen)
# ============================================================
print("=" * 70)
print("  STARTING TRAINING")
print("  Source: PA Q + [PRED]  →  Target: EN answer")
print("=" * 70)

model.train()

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

training_log  = []
global_step   = 0
accumulated   = 0
optimizer.zero_grad()
total_start   = time.time()
sample_times  = []

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()
    epoch_ce, epoch_jepa, epoch_cos, epoch_total = [], [], [], []

    epoch_samples = train_samples.copy()
    random.shuffle(epoch_samples)

    print(f"\n{'='*70}")
    print(f"  EPOCH {epoch+1}/{NUM_EPOCHS}")
    print(f"{'='*70}")

    for i, sample in enumerate(epoch_samples):
        t0 = time.time()

        # ---- BUILD PROMPTS ----
        pa_full_text, pa_qp_text, en_ans_text, pa_q_only = build_prompts(sample)
        pa_q_len = len(tokenizer.encode(pa_q_only))

        # ---- TOKENISE ----
        pa_full_enc = tokenizer(pa_full_text, return_tensors="pt",
                                truncation=True, max_length=MAX_LENGTH).to(device)
        pa_qp_enc   = tokenizer(pa_qp_text,   return_tensors="pt",
                                truncation=True, max_length=MAX_LENGTH).to(device)
        en_ans_enc  = tokenizer(en_ans_text,   return_tensors="pt",
                                truncation=True, max_length=MAX_LENGTH).to(device)

        # ---- LABELS (mask question, keep answer) ----
        labels = pa_full_enc["input_ids"].clone()
        mask_end = min(pa_q_len, labels.shape[1])
        labels[0, :mask_end] = -100

        # =========================================================
        # FORWARD PASS 1 — PA full (Q + answer) → CE loss
        # =========================================================
        outputs_ce = model(
            input_ids=pa_full_enc["input_ids"],
            attention_mask=pa_full_enc["attention_mask"],
            labels=labels,
        )
        ce_loss = outputs_ce.loss

        # =========================================================
        # FORWARD PASS 2 — PA question + [PRED] → h_pa
        #   Pred token only attends to PA question (causal).
        #   It must learn to PREDICT the EN answer representation.
        # =========================================================
        outputs_pa = model(
            input_ids=pa_qp_enc["input_ids"],
            attention_mask=pa_qp_enc["attention_mask"],
            output_hidden_states=True,
        )
        h_pa_pos = find_pred_token_pos(pa_qp_enc["input_ids"])
        h_pa = outputs_pa.hidden_states[-1][0, h_pa_pos, :]

        # =========================================================
        # FORWARD PASS 3 — EN answer (standalone) → h_en
        #   WITH gradient (both sides train jointly)
        # =========================================================
        outputs_en = model(
            input_ids=en_ans_enc["input_ids"],
            attention_mask=en_ans_enc["attention_mask"],
            output_hidden_states=True,
        )
        h_en_pos = find_en_hidden_pos(en_ans_enc["input_ids"])

        h_en = outputs_en.hidden_states[-1][0, h_en_pos, :]

        # =========================================================
        # COMPOSITE LOSS   L = γ·CE + λ·(1 − cos_sim(h_pa, h_en))
        # =========================================================
        cos_sim   = F.cosine_similarity(
            h_pa.unsqueeze(0).float(),
            h_en.unsqueeze(0).float(), dim=-1
        )
        jepa_loss  = 1.0 - cos_sim.mean()
        total_loss = GAMMA * ce_loss + LAMBDA_ * jepa_loss

        # ---- BACKWARD ----
        (total_loss / GRAD_ACCUM_STEPS).backward()
        accumulated += 1

        if accumulated >= GRAD_ACCUM_STEPS:
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            accumulated = 0
            global_step += 1

        # ---- RECORD ----
        dt = time.time() - t0
        sample_times.append(dt)
        cv, jv, cosv, tv = ce_loss.item(), jepa_loss.item(), cos_sim.item(), total_loss.item()
        epoch_ce.append(cv);  epoch_jepa.append(jv)
        epoch_cos.append(cosv); epoch_total.append(tv)

        training_log.append({
            "epoch": epoch + 1, "sample": i + 1,
            "global_sample": epoch * NUM_TRAIN + i + 1,
            "global_step": global_step,
            "ce_loss": round(cv, 6), "jepa_loss": round(jv, 6),
            "cosine_sim": round(cosv, 6), "total_loss": round(tv, 6),
            "lr": scheduler.get_last_lr()[0],
            "step_time_s": round(dt, 3),
        })

        if (i + 1) % 50 == 0 or i == 0 or (i + 1) == NUM_TRAIN:
            window = min(50, len(epoch_ce))
            avg_t = sum(sample_times[-window:]) / window
            rem   = (NUM_EPOCHS - epoch) * NUM_TRAIN - (i + 1)
            eta   = str(timedelta(seconds=int(rem * avg_t)))
            print(
                f"  [{i+1:3d}/{NUM_TRAIN}] "
                f"CE:{sum(epoch_ce[-window:])/window:.4f}  "
                f"JEPA:{sum(epoch_jepa[-window:])/window:.4f}  "
                f"CosSim:{sum(epoch_cos[-window:])/window:.4f}  "
                f"Total:{sum(epoch_total[-window:])/window:.4f}  "
                f"LR:{scheduler.get_last_lr()[0]:.1e}  "
                f"{avg_t:.2f}s/samp  ETA:{eta}"
            )

        if (i + 1) % 100 == 0:
            step_ckpt_path = f"{RESULTS_DIR}/checkpoint_epoch_{epoch+1}_step_{i+1}"
            print(f"\n  Saving step checkpoint to {step_ckpt_path}...")
            model.save_pretrained(step_ckpt_path)
            tokenizer.save_pretrained(step_ckpt_path)

        del outputs_ce, outputs_pa, outputs_en, h_pa, h_en, cos_sim, jepa_loss, total_loss, ce_loss
        torch.cuda.empty_cache()

    if accumulated > 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step(); scheduler.step(); optimizer.zero_grad()
        accumulated = 0; global_step += 1

    et = time.time() - epoch_start
    print(f"\n  ── Epoch {epoch+1} summary ──")
    print(f"     CE Loss   : {sum(epoch_ce)/len(epoch_ce):.4f}")
    print(f"     JEPA Loss : {sum(epoch_jepa)/len(epoch_jepa):.4f}")
    print(f"     CosSim    : {sum(epoch_cos)/len(epoch_cos):.4f}")
    print(f"     Total Loss: {sum(epoch_total)/len(epoch_total):.4f}")
    print(f"     Time      : {et:.0f}s  ({et/60:.1f} min)")
    print(f"     Opt. steps: {global_step}")

    epoch_ckpt_path = f"{RESULTS_DIR}/checkpoint_epoch_{epoch+1}"
    print(f"\n  Saving checkpoint to {epoch_ckpt_path}...")
    model.save_pretrained(epoch_ckpt_path)
    tokenizer.save_pretrained(epoch_ckpt_path)

tt = time.time() - total_start
print(f"\n{'='*70}")
print(f"  TRAINING COMPLETE — {tt:.0f}s ({tt/60:.1f} min)")
print(f"  Total optimizer steps: {global_step}")
print(f"{'='*70}")


  STARTING TRAINING (LLM-JEPA with predictor tokens)

  EPOCH 1/3
  [  1/500] CE:2.8789  JEPA:0.7823  CosSim:0.2177  Total:2.9571  LR:0.0e+00  1.56s/samp  ETA:0:38:59
  [ 50/500] CE:2.5177  JEPA:0.8670  CosSim:0.1330  Total:2.6044  LR:6.7e-06  1.35s/samp  ETA:0:32:44
  [100/500] CE:2.5053  JEPA:0.8751  CosSim:0.1249  Total:2.5929  LR:1.3e-05  1.34s/samp  ETA:0:31:18

  Saving step checkpoint to results/checkpoint_epoch_1_step_100...


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


  [150/500] CE:2.2170  JEPA:0.8790  CosSim:0.1210  Total:2.3049  LR:2.0e-05  1.36s/samp  ETA:0:30:30
  [200/500] CE:1.5858  JEPA:0.8528  CosSim:0.1472  Total:1.6711  LR:1.9e-05  1.35s/samp  ETA:0:29:15

  Saving step checkpoint to results/checkpoint_epoch_1_step_200...
  [250/500] CE:1.3085  JEPA:0.8319  CosSim:0.1681  Total:1.3917  LR:1.8e-05  1.35s/samp  ETA:0:28:11
  [300/500] CE:nan  JEPA:0.7837  CosSim:0.2163  Total:nan  LR:1.8e-05  1.36s/samp  ETA:0:27:15

  Saving step checkpoint to results/checkpoint_epoch_1_step_300...
  [350/500] CE:0.9598  JEPA:0.6718  CosSim:0.3282  Total:1.0270  LR:1.7e-05  1.36s/samp  ETA:0:26:04
  [400/500] CE:0.8444  JEPA:0.5517  CosSim:0.4483  Total:0.8996  LR:1.6e-05  1.35s/samp  ETA:0:24:48

  Saving step checkpoint to results/checkpoint_epoch_1_step_400...
  [450/500] CE:0.8026  JEPA:0.4180  CosSim:0.5820  Total:0.8444  LR:1.5e-05  1.35s/samp  ETA:0:23:35
  [500/500] CE:0.7660  JEPA:0.3177  CosSim:0.6823  Total:0.7978  LR:1.5e-05  1.35s/samp  ETA:0:

In [ ]:
# ============================================================
# POST-TRAINING: cosine similarity comparison
# ============================================================
print("Computing post-training cosine similarities...")
model.eval()
post_cos_sims = []

with torch.no_grad():
    for i in range(NUM_BASELINE):
        s = train_samples[i]
        _, pa_qp, en_ans, _ = build_prompts(s)

        pa_enc = tokenizer(pa_qp,  return_tensors="pt",
                           truncation=True, max_length=MAX_LENGTH).to(device)
        en_enc = tokenizer(en_ans, return_tensors="pt",
                           truncation=True, max_length=MAX_LENGTH).to(device)

        out_pa = model(**pa_enc, output_hidden_states=True)
        out_en = model(**en_enc, output_hidden_states=True)

        h_pa = out_pa.hidden_states[-1][0, find_pred_token_pos(pa_enc["input_ids"]), :]
        h_en = out_en.hidden_states[-1][0, find_en_hidden_pos(en_enc["input_ids"]), :]

        cs = F.cosine_similarity(
            h_pa.unsqueeze(0).float(),
            h_en.unsqueeze(0).float(), dim=-1
        ).item()
        post_cos_sims.append(cs)

post_mean = sum(post_cos_sims) / len(post_cos_sims)
post_std  = (sum((x - post_mean)**2 for x in post_cos_sims)
             / len(post_cos_sims)) ** 0.5

print(f"\n{'':18s} {'Pre-Train':>12s} {'Post-Train':>12s} {'Δ':>10s}")
print("-" * 56)
print(f"  {'Mean':12s}   {baseline_mean:10.4f}   {post_mean:10.4f}   {post_mean-baseline_mean:+8.4f}")
print(f"  {'Std':12s}   {baseline_std:10.4f}   {post_std:10.4f}")
print(f"  {'Min':12s}   {min(baseline_cos_sims):10.4f}   {min(post_cos_sims):10.4f}   {min(post_cos_sims)-min(baseline_cos_sims):+8.4f}")
print(f"  {'Max':12s}   {max(baseline_cos_sims):10.4f}   {max(post_cos_sims):10.4f}   {max(post_cos_sims)-max(baseline_cos_sims):+8.4f}")


Computing post-training cosine similarities...

                      Pre-Train   Post-Train          Δ
--------------------------------------------------------
  Mean               0.1213       0.9339    +0.8126
  Std                0.0406       0.0050
  Min                0.0480       0.9233    +0.8753
  Max                0.2794       0.9461    +0.6667


In [9]:
# ============================================================
# EVALUATION — 100 unseen Punjabi samples
# ============================================================
print("=" * 70)
print("  EVALUATION: 100 unseen Punjabi GSM8K samples")
print("=" * 70)

model.gradient_checkpointing_disable()   # not needed for inference
model.eval()

eval_logs  = []
correct    = 0
eval_start = time.time()

for i, sample in enumerate(test_samples):
    question  = sample["pa_question"]
    gold_raw  = sample["pa_answer"]
    gold      = extract_gold_answer(gold_raw)
    if gold is None:
        gold = extract_gold_answer(sample["en_answer"])

    prompt  = PROMPT_TEMPLATE.format(question=question)
    msgs    = [{"role": "user", "content": prompt}]
    text    = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt",
                       truncation=True, max_length=MAX_LENGTH).to(device)

    with torch.no_grad():
        gen = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=0.7, top_p=0.8, top_k=20,
            do_sample=True,
        )
    out_ids  = gen[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(out_ids, skip_special_tokens=True).strip()
    pred     = extract_model_answer(response)
    hit      = compare_answers(pred, gold)
    if hit:
        correct += 1

    eval_logs.append({
        "index": i, "pa_question": question,
        "en_question": sample["en_question"],
        "gold": gold, "pred": pred, "correct": hit,
        "response_preview": response[:400],
    })

    if (i + 1) % 10 == 0 or i == 0:
        print(f"  [{i+1:3d}/{NUM_TEST}]  "
              f"Gold:{str(gold):>6s}  Pred:{str(pred):>6s}  "
              f"{'✓' if hit else '✗'}   "
              f"Running: {correct}/{i+1} = {correct/(i+1)*100:.1f}%")

eval_time = time.time() - eval_start
accuracy  = correct / NUM_TEST * 100

print(f"\n{'='*70}")
print(f"  RESULT: {correct}/{NUM_TEST} = {accuracy:.1f}%")
print(f"  Time:   {eval_time:.0f}s  ({eval_time/NUM_TEST:.1f}s / sample)")
print(f"{'='*70}")


  EVALUATION: 100 unseen Punjabi GSM8K samples
  [  1/100]  Gold:   230  Pred:   230  ✓   Running: 1/1 = 100.0%
  [ 10/100]  Gold:     4  Pred:     0  ✗   Running: 3/10 = 30.0%
  [ 20/100]  Gold:    33  Pred:   0.4  ✗   Running: 4/20 = 20.0%
  [ 30/100]  Gold:    26  Pred:    30  ✗   Running: 8/30 = 26.7%
  [ 40/100]  Gold:    17  Pred:     9  ✗   Running: 12/40 = 30.0%
  [ 50/100]  Gold:     9  Pred:     7  ✗   Running: 14/50 = 28.0%


KeyboardInterrupt: 

In [ ]:
# ============================================================
# SAVE EVERYTHING
# ============================================================
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

summary = {
    "model": MODEL_NAME,
    "method": "Indic-JEPA composite loss SFT",
    "language": f"{LANG_NAME} ({LANG_CONFIG})",
    "timestamp": timestamp,
    "config": {
        "seed": SEED, "lr": LEARNING_RATE, "epochs": NUM_EPOCHS,
        "grad_accum": GRAD_ACCUM_STEPS, "max_length": MAX_LENGTH,
        "lora_rank": LORA_RANK, "lora_alpha": LORA_ALPHA,
        "gamma_ce": GAMMA, "lambda_jepa": LAMBDA_,
        "num_train": NUM_TRAIN, "num_test": NUM_TEST,
        "forward_passes_per_sample": 2,
    },
    "cosine_similarity": {
        "pre_training":  {"mean": round(baseline_mean, 4),
                          "std":  round(baseline_std, 4)},
        "post_training": {"mean": round(post_mean, 4),
                          "std":  round(post_std, 4)},
        "delta_mean":    round(post_mean - baseline_mean, 4),
    },
    "eval": {
        "correct": correct, "total": NUM_TEST,
        "accuracy": round(accuracy, 2),
        "time_s": round(eval_time, 1),
    },
    "training_time_s": round(tt, 1),
}

all_data = {
    "summary": summary,
    "training_log": training_log,
    "eval_logs": eval_logs,
    "baseline_cos_sims": [round(x, 6) for x in baseline_cos_sims],
    "post_cos_sims":     [round(x, 6) for x in post_cos_sims],
}

log_path = f"{RESULTS_DIR}/indic_jepa_train_{timestamp}.json"
sum_path = f"{RESULTS_DIR}/indic_jepa_summary_{timestamp}.json"

with open(log_path, "w", encoding="utf-8") as f:
    json.dump(all_data, f, ensure_ascii=False, indent=2)
with open(sum_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"Logs saved    → {log_path}")
print(f"Summary saved → {sum_path}")

# ---- Final report ----
print(f"\n{'='*70}")
print(f"  FINAL REPORT — Indic-JEPA on {MODEL_NAME}")
print(f"{'='*70}")
print(f"  Language:          {LANG_NAME}")
print(f"  Training samples:  {NUM_TRAIN} × {NUM_EPOCHS} epochs")
print(f"  Forward passes:    3 (PA full CE + PA Q+pred + EN answer)")
print(f"  Training time:     {tt/60:.1f} min")
print(f"")
print(f"  Cosine Similarity (PA ↔ EN hidden states):")
print(f"    Pre-training:    {baseline_mean:.4f}")
print(f"    Post-training:   {post_mean:.4f}  (Δ = {post_mean-baseline_mean:+.4f})")
print(f"")
print(f"  Evaluation (100 unseen PA samples):")
print(f"    Accuracy:        {correct}/{NUM_TEST} = {accuracy:.1f}%")
print(f"{'='*70}")

# Save LoRA adapter
adapter_path = f"{RESULTS_DIR}/lora_adapter_{timestamp}"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"\nLoRA adapter saved → {adapter_path}/")


Logs saved    → results/indic_jepa_train_20260415_201836.json
Summary saved → results/indic_jepa_summary_20260415_201836.json

  FINAL REPORT — Indic-JEPA on Qwen/Qwen3-1.7B
  Language:          Punjabi
  Training samples:  500 × 3 epochs
  Forward passes:    2 per sample (PA w/grad + EN frozen)
  Training time:     14.1 min

  Cosine Similarity (PA ↔ EN hidden states):
    Pre-training:    0.7672
    Post-training:   0.9607  (Δ = +0.1935)

  Evaluation (100 unseen PA samples):
    Accuracy:        34/100 = 34.0%

LoRA adapter saved → results/lora_adapter_20260415_201836/


In [ ]:
# ============================================================
# BASELINE SFT: Load a FRESH model (no JEPA-trained weights)
# ============================================================
import gc

# Free JEPA model memory
del model, optimizer, scheduler
gc.collect()
torch.cuda.empty_cache()

print("Loading fresh model for baseline SFT...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    use_cache=False,
)

lora_config_bl = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)
base_model = get_peft_model(base_model, lora_config_bl)
base_model.enable_input_require_grads()
base_model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)
base_model.print_trainable_parameters()
print("✓ Fresh model loaded for baseline SFT")


Loading fresh model for baseline SFT...


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable params: 17,432,576 || all params: 2,049,172,480 || trainable%: 0.8507
✓ Fresh model loaded for baseline SFT


In [ ]:
# ============================================================
# BASELINE TRAINING LOOP — CE ONLY, 1 forward pass / sample
# Same hyperparams, same data, same schedule
# ============================================================
print("=" * 70)
print("  STARTING BASELINE SFT (CE only, no JEPA)")
print("=" * 70)

BASELINE_RESULTS_DIR = "results_baseline_sft"
os.makedirs(BASELINE_RESULTS_DIR, exist_ok=True)

base_model.train()

bl_optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, base_model.parameters()),
    lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
)
bl_scheduler = get_linear_schedule_with_warmup(
    bl_optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

bl_training_log = []
bl_global_step  = 0
bl_accumulated  = 0
bl_optimizer.zero_grad()
bl_total_start  = time.time()
bl_sample_times = []

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()
    epoch_ce = []

    epoch_samples = train_samples.copy()
    random.shuffle(epoch_samples)

    print(f"\n{'='*70}")
    print(f"  EPOCH {epoch+1}/{NUM_EPOCHS}")
    print(f"{'='*70}")

    for i, sample in enumerate(epoch_samples):
        t0 = time.time()

        # ---- BUILD PROMPT (reuse existing helper) ----
        pa_full_text, pa_q_text, _ = build_prompts(sample)
        pa_q_len = len(tokenizer.encode(pa_q_text))

        # ---- TOKENISE ----
        pa_enc = tokenizer(pa_full_text, return_tensors="pt",
                           truncation=True, max_length=MAX_LENGTH).to(device)

        # ---- LABELS (mask question, keep answer — same as JEPA) ----
        labels = pa_enc["input_ids"].clone()
        mask_end = min(pa_q_len, labels.shape[1])
        labels[0, :mask_end] = -100

        # =========================================================
        # SINGLE FORWARD PASS — CE loss only (no English, no JEPA)
        # =========================================================
        outputs = base_model(
            input_ids=pa_enc["input_ids"],
            attention_mask=pa_enc["attention_mask"],
            labels=labels,
        )
        ce_loss = outputs.loss

        # ---- BACKWARD ----
        (ce_loss / GRAD_ACCUM_STEPS).backward()
        bl_accumulated += 1

        if bl_accumulated >= GRAD_ACCUM_STEPS:
            torch.nn.utils.clip_grad_norm_(base_model.parameters(), MAX_GRAD_NORM)
            bl_optimizer.step()
            bl_scheduler.step()
            bl_optimizer.zero_grad()
            bl_accumulated = 0
            bl_global_step += 1

        # ---- RECORD ----
        dt = time.time() - t0
        bl_sample_times.append(dt)
        cv = ce_loss.item()
        epoch_ce.append(cv)

        bl_training_log.append({
            "epoch": epoch + 1, "sample": i + 1,
            "global_sample": epoch * NUM_TRAIN + i + 1,
            "global_step": bl_global_step,
            "ce_loss": round(cv, 6),
            "lr": bl_scheduler.get_last_lr()[0],
            "step_time_s": round(dt, 3),
        })

        if (i + 1) % 50 == 0 or i == 0 or (i + 1) == NUM_TRAIN:
            window = min(50, len(epoch_ce))
            avg_t = sum(bl_sample_times[-window:]) / window
            rem   = (NUM_EPOCHS - epoch) * NUM_TRAIN - (i + 1)
            eta   = str(timedelta(seconds=int(rem * avg_t)))
            print(
                f"  [{i+1:3d}/{NUM_TRAIN}] "
                f"CE:{sum(epoch_ce[-window:])/window:.4f}  "
                f"LR:{bl_scheduler.get_last_lr()[0]:.1e}  "
                f"{avg_t:.2f}s/samp  ETA:{eta}"
            )

        # ---- SAVE CHECKPOINT (EVERY 100 SAMPLES) ----
        if (i + 1) % 100 == 0:
            step_ckpt_path = f"{BASELINE_RESULTS_DIR}/checkpoint_epoch_{epoch+1}_step_{i+1}"
            print(f"\n  Saving step checkpoint to {step_ckpt_path}...")
            base_model.save_pretrained(step_ckpt_path)
            tokenizer.save_pretrained(step_ckpt_path)
            
        del outputs, ce_loss
        torch.cuda.empty_cache()

    # Flush remaining
    if bl_accumulated > 0:
        torch.nn.utils.clip_grad_norm_(base_model.parameters(), MAX_GRAD_NORM)
        bl_optimizer.step(); bl_scheduler.step(); bl_optimizer.zero_grad()
        bl_accumulated = 0; bl_global_step += 1

    et = time.time() - epoch_start
    print(f"\n  ── Epoch {epoch+1} summary ──")
    print(f"     CE Loss   : {sum(epoch_ce)/len(epoch_ce):.4f}")
    print(f"     Time      : {et:.0f}s  ({et/60:.1f} min)")
    print(f"     Opt. steps: {bl_global_step}")

    # ---- SAVE CHECKPOINT (END OF EPOCH) ----
    epoch_ckpt_path = f"{BASELINE_RESULTS_DIR}/checkpoint_epoch_{epoch+1}"
    print(f"\n  Saving epoch checkpoint to {epoch_ckpt_path}...")
    base_model.save_pretrained(epoch_ckpt_path)
    tokenizer.save_pretrained(epoch_ckpt_path)

bl_tt = time.time() - bl_total_start
print(f"\n{'='*70}")
print(f"  BASELINE TRAINING COMPLETE — {bl_tt:.0f}s ({bl_tt/60:.1f} min)")
print(f"  Total optimizer steps: {bl_global_step}")
print(f"{'='*70}")


  STARTING BASELINE SFT (CE only, no JEPA)

  EPOCH 1/3
  [  1/500] CE:1.7533  LR:0.0e+00  0.44s/samp  ETA:0:11:06
  [ 50/500] CE:2.5491  LR:6.7e-06  0.45s/samp  ETA:0:10:54
  [100/500] CE:2.4177  LR:1.3e-05  0.45s/samp  ETA:0:10:34

  Saving step checkpoint to results_baseline_sft/checkpoint_epoch_1_step_100...
  [150/500] CE:2.1859  LR:2.0e-05  0.46s/samp  ETA:0:10:18
  [200/500] CE:1.5580  LR:1.9e-05  0.45s/samp  ETA:0:09:48

  Saving step checkpoint to results_baseline_sft/checkpoint_epoch_1_step_200...
  [250/500] CE:1.2396  LR:1.8e-05  0.46s/samp  ETA:0:09:31
  [300/500] CE:1.0755  LR:1.8e-05  0.45s/samp  ETA:0:09:04

  Saving step checkpoint to results_baseline_sft/checkpoint_epoch_1_step_300...
  [350/500] CE:0.9715  LR:1.7e-05  0.46s/samp  ETA:0:08:43
  [400/500] CE:nan  LR:1.6e-05  0.45s/samp  ETA:0:08:17

  Saving step checkpoint to results_baseline_sft/checkpoint_epoch_1_step_400...
  [450/500] CE:0.8263  LR:1.5e-05  0.46s/samp  ETA:0:08:04
  [500/500] CE:0.7543  LR:1.5e-05

In [ ]:
# ============================================================
# BASELINE EVALUATION — same 100 unseen samples
# ============================================================
print("=" * 70)
print("  BASELINE EVALUATION: 100 unseen Punjabi GSM8K samples")
print("=" * 70)

base_model.gradient_checkpointing_disable()
base_model.eval()

bl_eval_logs = []
bl_correct   = 0
bl_eval_start = time.time()

for i, sample in enumerate(test_samples):
    question = sample["pa_question"]
    gold_raw = sample["pa_answer"]
    gold     = extract_gold_answer(gold_raw)
    if gold is None:
        gold = extract_gold_answer(sample["en_answer"])

    prompt = PROMPT_TEMPLATE.format(question=question)
    msgs   = [{"role": "user", "content": prompt}]
    text   = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt",
                       truncation=True, max_length=MAX_LENGTH).to(device)

    with torch.no_grad():
        gen = base_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=0.7, top_p=0.8, top_k=20,
            do_sample=True,
        )
    out_ids  = gen[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(out_ids, skip_special_tokens=True).strip()
    pred     = extract_model_answer(response)
    hit      = compare_answers(pred, gold)
    if hit:
        bl_correct += 1

    bl_eval_logs.append({
        "index": i, "pa_question": question,
        "en_question": sample["en_question"],
        "gold": gold, "pred": pred, "correct": hit,
        "response_preview": response[:400],
    })

    if (i + 1) % 10 == 0 or i == 0:
        print(f"  [{i+1:3d}/{NUM_TEST}]  "
              f"Gold:{str(gold):>6s}  Pred:{str(pred):>6s}  "
              f"{'✓' if hit else '✗'}   "
              f"Running: {bl_correct}/{i+1} = {bl_correct/(i+1)*100:.1f}%")

bl_eval_time = time.time() - bl_eval_start
bl_accuracy  = bl_correct / NUM_TEST * 100

print(f"\n{'='*70}")
print(f"  BASELINE RESULT: {bl_correct}/{NUM_TEST} = {bl_accuracy:.1f}%")
print(f"  Time: {bl_eval_time:.0f}s  ({bl_eval_time/NUM_TEST:.1f}s / sample)")
print(f"{'='*70}")


  BASELINE EVALUATION: 100 unseen Punjabi GSM8K samples
  [  1/100]  Gold:   230  Pred:   230  ✓   Running: 1/1 = 100.0%
  [ 10/100]  Gold:     4  Pred: 2.875  ✗   Running: 3/10 = 30.0%
  [ 20/100]  Gold:    33  Pred: 0.644  ✗   Running: 7/20 = 35.0%
  [ 30/100]  Gold:    26  Pred:    30  ✗   Running: 12/30 = 40.0%
  [ 40/100]  Gold:    17  Pred:    15  ✗   Running: 16/40 = 40.0%
  [ 50/100]  Gold:     9  Pred:    -1  ✗   Running: 18/50 = 36.0%
  [ 60/100]  Gold:     9  Pred:     9  ✓   Running: 23/60 = 38.3%


KeyboardInterrupt: 

In [ ]:
# ============================================================
# SAVE BASELINE & COMPARE WITH JEPA
# ============================================================
bl_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

bl_summary = {
    "model": MODEL_NAME,
    "method": "Baseline SFT (CE only, no JEPA)",
    "language": f"{LANG_NAME} ({LANG_CONFIG})",
    "timestamp": bl_timestamp,
    "config": {
        "seed": SEED, "lr": LEARNING_RATE, "epochs": NUM_EPOCHS,
        "grad_accum": GRAD_ACCUM_STEPS, "max_length": MAX_LENGTH,
        "lora_rank": LORA_RANK, "lora_alpha": LORA_ALPHA,
        "loss": "CE only",
        "num_train": NUM_TRAIN, "num_test": NUM_TEST,
        "forward_passes_per_sample": 1,
    },
    "eval": {
        "correct": bl_correct, "total": NUM_TEST,
        "accuracy": round(bl_accuracy, 2),
        "time_s": round(bl_eval_time, 1),
    },
    "training_time_s": round(bl_tt, 1),
}

bl_all_data = {
    "summary": bl_summary,
    "training_log": bl_training_log,
    "eval_logs": bl_eval_logs,
}

bl_log_path = f"{BASELINE_RESULTS_DIR}/baseline_sft_train_{bl_timestamp}.json"
bl_sum_path = f"{BASELINE_RESULTS_DIR}/baseline_sft_summary_{bl_timestamp}.json"

with open(bl_log_path, "w", encoding="utf-8") as f:
    json.dump(bl_all_data, f, ensure_ascii=False, indent=2)
with open(bl_sum_path, "w", encoding="utf-8") as f:
    json.dump(bl_summary, f, ensure_ascii=False, indent=2)

# Save baseline LoRA adapter
bl_adapter_path = f"{BASELINE_RESULTS_DIR}/lora_adapter_{bl_timestamp}"
base_model.save_pretrained(bl_adapter_path)
tokenizer.save_pretrained(bl_adapter_path)

print(f"Baseline logs    → {bl_log_path}")
print(f"Baseline summary → {bl_sum_path}")
print(f"Baseline adapter → {bl_adapter_path}/")

# ============================================================
# HEAD-TO-HEAD COMPARISON
# ============================================================
# Pull JEPA results from earlier cells (already in runtime)
jepa_accuracy = accuracy   # from Cell 9
jepa_time     = tt         # from Cell 7

print(f"\n{'='*70}")
print(f"  HEAD-TO-HEAD: JEPA vs Baseline SFT")
print(f"{'='*70}")
print(f"  {'':30s} {'JEPA SFT':>12s} {'Baseline':>12s} {'Δ':>8s}")
print(f"  {'-'*64}")
print(f"  {'Eval Accuracy (%)':30s} {jepa_accuracy:>11.1f}% {bl_accuracy:>11.1f}% {bl_accuracy - jepa_accuracy:>+7.1f}%")
print(f"  {'Training Time (min)':30s} {jepa_time/60:>11.1f}  {bl_tt/60:>11.1f}  {bl_tt/60 - jepa_time/60:>+7.1f}")
print(f"  {'Fwd Passes / Sample':30s} {'2':>12s} {'1':>12s}")
print(f"  {'Loss':30s} {'CE+CosSim':>12s} {'CE only':>12s}")
print(f"  {'CosSim Δ (post−pre)':30s} {post_mean - baseline_mean:>+11.4f}  {'N/A':>12s}")
print(f"{'='*70}")
